# Traductor RNN - Seq2Seq LSTM
## Universidad Autónoma del Caribe (UAC)
### Metodología CRISP-ML(Q)

# FASE 1: Business & Data Understanding
## 1.1 Definición del Problema
- **Objetivo**: Traductor automático Inglés ↔ Español
- **Arquitectura**: Seq2Seq LSTM (Red Neuronal Recurrente)
- **Métricas**: BLEU Score ≥ 0.30
- **Latencia**: < 2 segundos

In [ ]:
# ========================================
# CELDA 1: Instalación de dependencias
# ========================================

!pip install torch numpy gradio

In [ ]:
# ========================================
# CELDA 2: Importar librerías
# ========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import gradio as gr

# Determinar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

## 1.2 Análisis de Requisitos
- Corpus paralelo inglés-español
- Vocabulario mínimo de 100+ frases
- Tokenización a nivel de palabra

# FASE 2: Data Preparation
## 2.1 Corpus de Entrenamiento

In [ ]:
# ========================================
# CELDA 3: Definir corpus de entrenamiento
# ========================================

# Corpus amplio (322 parejas inglés-español)
CORPUS = [
    # Saludos básicos
    ("hello", "hola"),
    ("goodbye", "adios"),
    ("good morning", "buenos dias"),
    ("good night", "buenas noches"),
    ("good evening", "buenas tardes"),
    ("see you later", "hasta luego"),
    
    # Cortesía
    ("thank you", "gracias"),
    ("thank you very much", "muchas gracias"),
    ("please", "por favor"),
    ("you are welcome", "de nada"),
    ("excuse me", "disculpe"),
    ("sorry", "lo siento"),
    
    # Respuestas
    ("yes", "si"),
    ("no", "no"),
    ("maybe", "quizas"),
    ("of course", "por supuesto"),
    
    # Pronombres
    ("i", "yo"),
    ("you", "tu"),
    ("he", "el"),
    ("she", "ella"),
    ("we", "nosotros"),
    ("they", "ellos"),
    
    # Familia
    ("father", "padre"),
    ("mother", "madre"),
    ("brother", "hermano"),
    ("sister", "hermana"),
    
    # Universidad
    ("university", "universidad"),
    ("class", "clase"),
    ("professor", "profesor"),
    ("student", "estudiante"),
    ("exam", "examen"),
    ("homework", "tarea"),
    ("book", "libro"),
    ("computer", "computadora"),
    
    # Oraciones útiles
    ("i am a student", "soy estudiante"),
    ("where is the library", "donde esta la biblioteca"),
    ("the exam is difficult", "el examen es dificil"),
    ("i need to study", "necesito estudiar"),
    ("how are you", "como estas"),
    ("i study at the university", "estudio en la universidad"),
    ("you are a teacher", "tu eres maestro"),
    ("he is a professor", "el es profesor"),
    ("she is a student", "ella es estudiante"),
    ("we are friends", "somos amigos"),
    ("what is your name", "cual es tu nombre"),
    ("my name is john", "me llamo john"),
    ("nice to meet you", "mucho gusto"),
    ("the class starts at eight", "la clase empieza a las ocho"),
    ("i need a book", "necesito un libro"),
    ("the professor is strict", "el profesor es estricto"),
    ("i have a class at nine", "tengo clase a las nueve"),
    ("the lecture is interesting", "la conferencia es interesante"),
    ("when is the exam", "cuando es el examen"),
    ("i passed the exam", "aprobe el examen"),
    ("i am late for class", "llegue tarde a clase"),
    
    # Números
    ("one", "uno"),
    ("two", "dos"),
    ("three", "tres"),
    ("four", "cuatro"),
    ("five", "cinco"),
    ("six", "seis"),
    ("seven", "siete"),
    ("eight", "ocho"),
    ("nine", "nueve"),
    ("ten", "diez"),
    
    # Días de la semana
    ("monday", "lunes"),
    ("tuesday", "martes"),
    ("wednesday", "miercoles"),
    ("thursday", "jueves"),
    ("friday", "viernes"),
    ("saturday", "sabado"),
    ("sunday", "domingo"),
    ("today", "hoy"),
    ("tomorrow", "manana"),
    
    # Adjetivos
    ("good", "bueno"),
    ("bad", "malo"),
    ("big", "grande"),
    ("small", "pequeno"),
    ("new", "nuevo"),
    ("old", "viejo"),
    ("fast", "rapido"),
    ("slow", "lento"),
    ("easy", "facil"),
    ("difficult", "dificil"),
    
    # Preguntas
    ("how much", "cuanto"),
    ("what time is it", "que hora es"),
]

# Añadir inversión del corpus (ES -> EN)
for es, en in list(CORPUS):
    if (en, es) not in CORPUS:
        CORPUS.append((en, es))

print(f"Total corpus: {len(CORPUS)} parejas")

## 2.2 Vocabulario y Tokenización

In [ ]:
# ========================================
# CELDA 4: Crear vocabulario
# ========================================

# Tokens especiales
PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

class Vocab:
    def __init__(self):
        self.w2i = {PAD: 0, UNK: 1, SOS: 2, EOS: 3}
        self.i2w = {0: PAD, 1: UNK, 2: SOS, 3: EOS}
        self.n = 4

    def add(self, text):
        for w in text.lower().split():
            if w not in self.w2i:
                self.w2i[w] = self.n
                self.i2w[self.n] = w
                self.n += 1

    def encode(self, text, max_len, sos=False, eos=False):
        ids = []
        if sos: ids.append(self.w2i[SOS])
        for w in text.lower().split():
            ids.append(self.w2i.get(w, self.w2i[UNK]))
        if eos: ids.append(self.w2i[EOS])
        while len(ids) < max_len: ids.append(self.w2i[PAD])
        return ids[:max_len]

    def decode(self, ids):
        ws = []
        for i in ids:
            if torch.is_tensor(i): i = i.item()
            w = self.i2w.get(i, UNK)
            if w not in [PAD, SOS, EOS]: ws.append(w)
        return " ".join(ws)

# Crear vocabularios
src_v, tgt_v = Vocab(), Vocab()
for s, t in CORPUS:
    src_v.add(s)
    tgt_v.add(t)

print(f"Vocabulario src: {src_v.n} palabras")
print(f"Vocabulario tgt: {tgt_v.n} palabras")

# FASE 3: Modeling
## 3.1 Arquitectura Seq2Seq LSTM

In [ ]:
# ========================================
# CELDA 5: Definir Encoder
# ========================================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, 
                         batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

In [ ]:
# ========================================
# CELDA 6: Definir Decoder
# ========================================

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, 
                         batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, cell):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(outputs.squeeze(1))
        return prediction, hidden, cell

In [ ]:
# ========================================
# CELDA 7: Definir Modelo Seq2Seq
# ========================================

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(src.device)
        
        _, hidden, cell = self.encoder(src)
        decoder_input = tgt[:, 0]  # SOS token
        
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(
                decoder_input.unsqueeze(1), hidden, cell
            )
            outputs[:, t] = output
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = tgt[:, t] if teacher_force else top1
        
        return outputs

In [ ]:
# ========================================
# CELDA 8: Inicializar modelo
# ========================================

# Hyperparámetros
EMBED_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3
MAX_LEN = 20

# Crear modelo
encoder = Encoder(src_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
decoder = Decoder(tgt_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
model = Seq2Seq(encoder, decoder).to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {total_params:,}")

# FASE 4: Training
## 4.1 Preparación de Datos

In [ ]:
# ========================================
# CELDA 9: Crear Dataset
# ========================================

class TranslationDataset(Dataset):
    def __init__(self, data, src_vocab, tgt_vocab, max_len):
        self.data = [
            (src_vocab.encode(s, max_len), 
             tgt_vocab.encode(t, max_len, sos=True, eos=True)) 
            for s, t in data
        ]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), torch.tensor(self.data[idx][1])

# Crear dataset y dataloader
dataset = TranslationDataset(CORPUS, src_v, tgt_v, MAX_LEN)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset: {len(dataset)} muestras")
print(f"Batches: {len(dataloader)}")

In [ ]:
# ========================================
# CELDA 10: Configurar entrenamiento
# ========================================

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
losses = []

print(f"Épocas: {EPOCHS}")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Loss: CrossEntropyLoss")

## 4.2 Bucle de Entrenamiento

In [ ]:
# ========================================
# CELDA 11: Entrenar modelo
# ========================================

model.train()
print("Iniciando entrenamiento...")

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0
    
    for src, tgt in dataloader:
        src = src.to(device)
        tgt = tgt.to(device)
        
        optimizer.zero_grad()
        
        output = model(src, tgt)
        output = output.view(-1, output.shape[-1])
        tgt = tgt.view(-1)
        
        loss = criterion(output, tgt)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f}")

print("¡Entrenamiento completado!")

## 4.3 Curvas de Aprendizaje

In [ ]:
# ========================================
# CELDA 12: Graficar pérdida
# ========================================

plt.figure(figsize=(10, 5))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Pérdida', fontsize=12)
plt.title('Curva de Aprendizaje - Loss vs Época', fontsize=14)
plt.grid(True, alpha=0.3)
plt.savefig('training_loss.png', dpi=150)
plt.show()

print(f"Última pérdida: {losses[-1]:.4f}")

# FASE 5: Evaluation
## 5.1 Métrica BLEU Score

In [ ]:
# ========================================
# CELDA 13: Función BLEU Score
# ========================================

def calculate_bleu(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    
    if not hyp_words:
        return 0.0
    
    matches = sum(1 for w in hyp_words if w in ref_words)
    precision = matches / len(hyp_words)
    
    brevity_penalty = min(1.0, np.exp(1 - len(ref_words) / max(len(hyp_words), 1))
    
    return brevity_penalty * precision

print("Función BLEU definida")

In [ ]:
# ========================================
# CELDA 14: Evaluar modelo
# ========================================

model.eval()

test_sentences = [
    ("hello", "hola"),
    ("thank you", "gracias"),
    ("i am a student", "soy estudiante"),
    ("where is the library", "donde esta la biblioteca"),
    ("the exam is difficult", "el examen es dificil"),
    ("good morning", "buenos dias"),
    ("how are you", "como estas"),
    ("i study at the university", "estudio en la universidad"),
]

total_bleu = 0
results = []

with torch.no_grad():
    for src_text, tgt_text in test_sentences:
        # Encode fuente
        src_encoded = torch.tensor([src_v.encode(src_text, MAX_LEN)]).to(device)
        
        # Encoder
        _, hidden, cell = encoder(src_encoded)
        
        # Decodificar
        decoder_input = torch.tensor([tgt_v.w2i[SOS]]).to(device)
        result_ids = []
        
        for _ in range(MAX_LEN):
            output, hidden, cell = decoder(
                decoder_input.unsqueeze(1), hidden, cell
            )
            top_token = output.argmax(1).item()
            
            if top_token == tgt_v.w2i[EOS] or top_token == tgt_v.w2i[PAD]:
                break
            
            result_ids.append(top_token)
            decoder_input = torch.tensor([top_token]).to(device)
        
        translated_text = tgt_v.decode(result_ids)
        bleu_score = calculate_bleu(tgt_text, translated_text)
        total_bleu += bleu_score
        
        results.append((src_text, tgt_text, translated_text, bleu_score))
        print(f"{src_text} -> {translated_text} (BLEU: {bleu_score:.2f})")

avg_bleu = total_bleu / len(test_sentences)
print(f"\n=== BLEU Score Promedio: {avg_bleu:.2f} ===")

# FASE 6: Deployment
## 6.1 Guardar Modelo

In [ ]:
# ========================================
# CELDA 15: Guardar modelo entrenado
# ========================================

torch.save({
    'model_state_dict': model.state_dict(),
    'encoder_state_dict': encoder.state_dict(),
    'decoder_state_dict': decoder.state_dict(),
    'src_vocab': src_v.w2i,
    'tgt_vocab': tgt_v.w2i,
    'src_idx2word': src_v.i2w,
    'tgt_idx2word': tgt_v.i2w,
    'max_len': MAX_LEN,
    'bleu_score': avg_bleu,
}, 'translator_model.pt')

print("Modelo guardado: translator_model.pt")

## 6.2 Interfaz Gradio

In [ ]:
# ========================================
# CELDA 16: Interfaz de traducción Gradio
# ========================================

def translate_text(text, direction="EN->ES"):
    if not text.strip():
        return ""
    
    model.eval()
    
    with torch.no_grad():
        if direction == "ES->EN":
            src_vocab = tgt_v
            tgt_vocab = src_v
        else:
            src_vocab = src_v
            tgt_vocab = tgt_v
        
        src_encoded = torch.tensor([src_vocab.encode(text, MAX_LEN)]).to(device)
        _, hidden, cell = encoder(src_encoded)
        
        decoder_input = torch.tensor([tgt_vocab.w2i[SOS]]).to(device)
        result_ids = []
        
        for _ in range(MAX_LEN):
            output, hidden, cell = decoder(
                decoder_input.unsqueeze(1), hidden, cell
            )
            top_token = output.argmax(1).item()
            
            if top_token == tgt_vocab.w2i[EOS] or top_token == tgt_vocab.w2i[PAD]:
                break
            
            result_ids.append(top_token)
            decoder_input = torch.tensor([top_token]).to(device)
        
        return tgt_vocab.decode(result_ids)

# Crear interfaz Gradio
with gr.Blocks() as demo:
    gr.Markdown("# Traductor RNN - Universidad Autónoma del Caribe")
    gr.Markdown("## Seq2Seq LSTM - Metodología CRISP-ML(Q)")
    gr.Markdown(f"BLEU Score: {avg_bleu:.2f}")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Texto a traducir")
            direction = gr.Radio(["EN->ES", "ES->EN"], label="Dirección", value="EN->ES")
            translate_btn = gr.Button("Traducir")
        
        with gr.Column():
            output_text = gr.Textbox(label="Traducción")
    
    translate_btn.click(fn=translate_text, inputs=[input_text, direction], outputs=output_text)

demo.launch()

## Resumen del Proyecto
- **Fase 1**: Business & Data Understanding ✓
- **Fase 2**: Data Preparation ✓
- **Fase 3**: Modeling (Seq2Seq LSTM) ✓
- **Fase 4**: Training ✓
- **Fase 5**: Evaluation (BLEU Score) ✓
- **Fase 6**: Deployment (Gradio) ✓